In [2]:
import sys
sys.path.append('..')

In [3]:
import geopandas as gpd
import matplotlib.pyplot as plt
import rioxarray
import numpy as np
import xarray as xr
from rasterstats import zonal_stats, point_query
from geocube.api.core import make_geocube
import pandas as pd
import seaborn

In [4]:
df_lst_ndvi = pd.read_parquet('../data/processed/20190729_20200827/lst_ndvi.parquet.gzip')
gdf = gpd.read_file('../data/processed/20190729_20200827/temperature.geojson')
gdf["nb_id"] = gdf["nb_id"].astype(int)
gdf_json = gdf.set_index("nb_id").__geo_interface__

In [5]:
import plotly.graph_objects as go

In [6]:
COLS_DISPLAY= [
    "ntaname", "boroname",     
    "lst_mean", "lst_std", 
    "lst_min", "lst_max", 

    "ndvi_mean", "ndvi_std",
    "ndvi_min", "ndvi_max",
    ]

In [7]:
# fig = go.Figure(data=go.Choroplethmapbox(
#     geojson=gdf_json,
#     locations=gdf["nb_id"],
#     featureidkey="id",
#     z=gdf["lst_mean"],
#     colorscale="RdYlBu_r",
#     marker_line_color="red",   # borders between neighborhoods
#     marker_line_width=1,
#     marker_opacity=0.7,          # let the basemap show through (explore() look)
#     colorbar=dict(title=dict(text="LST (°C)")),
#     customdata=gdf[COLS_DISPLAY],
#     hovertemplate=(
#             "<b>%{customdata[0]}</b> (%{customdata[1]})<br>"
#             "type : %{customdata[2]}<br> <br>"

#             "LST: <br>"
#             "       mean:  %{customdata[3]:.1f}°C<br>"
#             "       std:   %{customdata[4]:.1f}°C<br>"
#             "       min:   %{customdata[5]:.1f}°C<br>"
#             "       max:   %{customdata[6]:.1f}°C<br> <br>"

#             "NDVI: <br>"
#             "       mean: %{customdata[7]:.2f}<br>"
#             "       std:  %{customdata[8]:.2f}<br>"
#             "       min:  %{customdata[9]:.2f}<br>"
#             "       max:  %{customdata[10]:.2f}<extra></extra>"

#     ),
    
# ))

# # fig.update_geos(fitbounds="locations", visible=True, projection_type='mercator', map_style='carto-voyager')
# fig.update_layout(
#     margin=dict(t=0, b=0, l=0, r=0),
#     mapbox=dict(
#         style="open-street-map",
#         center=dict(lat=40.70, lon=-73.95),
#         zoom=9,
#     ),
# )


# fig.show()


In [24]:
import plotly.express as px

In [59]:
nbins=80
counts, xedges, yedges = np.histogram2d(
    df_lst_ndvi["ndvi"], df_lst_ndvi["lst"], bins=nbins
)
counts = counts.T                                   # heatmap wants [y, x]
z = np.where(counts > 0, np.log10(counts), np.nan)  # log; empty bins → NaN

xc = 0.5 * (xedges[:-1] + xedges[1:])               # bin centers
yc = 0.5 * (yedges[:-1] + yedges[1:])

fig = go.Figure()
fig.add_trace(go.Heatmap(
    x=xc, y=yc, z=z,
    colorscale="Blues_r",
    colorbar=dict(
        title=dict(text="pixel count"),
        tickvals=[0, 1, 2, 3],                      # log10 values…
        ticktext=[
            "10<sup>0</sup>", 
            "10<sup>1</sup>", 
            "10<sup>2</sup>", 
            "10<sup>3</sup>"],       # …shown as real counts
    ),
    hovertemplate=
    """ 
    Pixel Count: 10<sup>%{z:.1f}</sup><br>
    NDVI: %{x:.2f}<br>
    LST: %{y:.1f}°C<extra></extra>
    """,
))

x = np.linspace(0,1, 100)
fig.add_trace(go.Scatter(
    x=x, y=slope * x + intercept, mode="lines",
    line=dict(color="crimson", width=2, dash="dash"),
    name=f"fit (r = {pearson:.2f})",
))

fig.update_layout(
    width=500,
    height=400,
    margin=dict(t=50, b=50, l=60, r=20),
    title=dict(text=f"Greener blocks stay cooler — {slope / 10:.2f} °C per 0.1 NDVI"),
    xaxis_title="NDVI (vegetation)",
    yaxis_title="LST (°C)",
    legend=dict(x=0.98, y=0.98, xanchor="right", yanchor="top"),
    template="plotly_white",
)

/var/folders/k8/8bd2c6nd31s1yp4rsc4gj7340000gn/T/ipykernel_22949/3961158477.py:6: RuntimeWarning: divide by zero encountered in log10
  z = np.where(counts > 0, np.log10(counts), np.nan)  # log; empty bins → NaN
